Trains a simple VAE on MNIST to generate images of hand drawn digits. Designed to be run locally on CPU

In [1]:
# Main imports
import torch as t
from dataclasses import dataclass
from tqdm.notebook import tqdm
import wandb
from torch.utils.data import DataLoader
from jaxtyping import Float
from torch import Tensor, nn
import einops
import sys
from plotly.express import imshow

In [2]:
# Local imports
# Running locally from training/: make the repo root importable
sys.path.append("..")
from dataloaders import mnist
from models.mnist_vae import VAE

In [3]:
# Check if we have access to a GPU
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
print(device)

cpu


In [4]:
def display_data(x: Tensor, nrows: int, title: str) -> None:
    """Displays a batch of data, using plotly."""
    ncols = x.shape[0] // nrows
    # Reshape into the right shape for plotting (make it 2D if image is monochrome)
    y = einops.rearrange(x, "(b1 b2) c h w -> (b1 h) (b2 w) c", b1=nrows).squeeze()
    # Normalize in the 0-1 range, then map to integer type
    y = (y - y.min()) / (y.max() - y.min())
    y = (y * 255).to(dtype=t.uint8)
    # Display data
    imshow(
        y,
        binary_string=(y.ndim == 2),
        height=50 * (nrows + 4),
        width=50 * (ncols + 5),
        title=f"{title}<br>single input shape = {x[0].shape}",
    ).show()

In [5]:
@dataclass
class VAEArgs:
    # architecture
    latent_dim_size: int = 5   # If edited, model can not be run in UI
    hidden_dim_size: int = 128 # If edited, model can not be run in UI

    # data / training
    trainset_size: int = 60_000
    batch_size: int = 512
    epochs: int = 10
    lr: float = 1e-3
    betas: tuple[float, float] = (0.5, 0.999)
    beta_kl: float = 0.1

    # logging
    use_wandb: bool = True
    wandb_project: str | None = "mnist-vae"
    wandb_name: str | None = None
    log_every_n_steps: int = 250

In [6]:
class VAETrainer:
    def __init__(self, args: VAEArgs):
        self.args = args
        self.trainset, testset = mnist.get_mnist(trainset_size=args.trainset_size, testset_size=100)
        # One test image of each digit, 0-9, reconstructed at every log
        holdout = {}
        for img, label in testset:
            holdout.setdefault(label, img)
            if len(holdout) == 10:
                break
        self.holdout = t.stack([holdout[d] for d in range(10)]).to(device)   # (10, 1, 28, 28)
        self.trainloader = DataLoader(self.trainset, batch_size=args.batch_size, shuffle=True, num_workers=0)
        
        self.model = VAE(
            latent_dim_size=args.latent_dim_size,
            hidden_dim_size=args.hidden_dim_size,
        ).to(device)
        self.optimizer = t.optim.Adam(self.model.parameters(), lr=args.lr, betas=args.betas)

    def training_step(
        self, img: Float[Tensor, "batch 1 height width"]
    ) -> Float[Tensor, ""]:
        """
        Performs a training step on the batch of images in `img`. Returns the loss. Logs to wandb
        if enabled.
        """
        # Forward pass (1 batch)
        decoded_img, mu, logsigma = self.model(img)

        # Loss: difference between img | output, AND KL divergence
        kl_divergence = ((t.exp(logsigma)**2 + mu**2 - 1) / 2 - logsigma).mean() * self.args.beta_kl
        reconstruct_to_img_loss = nn.MSELoss()(decoded_img, img)
        loss =  reconstruct_to_img_loss + kl_divergence

        # Back pass
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()

        # Logging (loss) (increment step)
        self.step += 1

        if self.args.use_wandb:
            wandb.log(
                dict(
                    reconstruction_loss=reconstruct_to_img_loss.item(),
                    kl_div_loss=kl_divergence.item(),
                    mean=mu.mean().item(),
                    std=t.exp(logsigma).mean().item(),
                    total_loss=loss.item(),
                ),
                step=self.step)

        return loss

    @t.inference_mode()
    def log_samples(self) -> None:
        """
        Evaluates model on holdout data, either logging to wandb or displaying output inline.
        """
        assert self.step > 0, "First call should come after a training step. Remember to increment `self.step`."
        output = self.model(self.holdout)[0]
        if self.args.use_wandb:
            output = (output - output.min()) / (output.max() - output.min())  # Normalize to [0, 1]
            output = (output * 255).to(dtype=t.uint8)  # Convert to uint8 for logging
            wandb.log({"images": [wandb.Image(arr) for arr in output.cpu().numpy()]}, step=self.step)
        else:
            display_data(t.concat([self.holdout, output]), nrows=2, title="VAE reconstructions")

    def train(self) -> VAE:
        """Performs a full training run."""
        self.step = 0
        if self.args.use_wandb:
            wandb.init(project=self.args.wandb_project, name=self.args.wandb_name)
            wandb.watch(self.model)

        # Loop over epochs
        for epoch in range(self.args.epochs):
            progress_bar = tqdm(self.trainloader, ascii=True)
            # Loop over batches
            for images, labels in progress_bar:
                # Train step
                images = images.to(device)
                loss = self.training_step(images)
                progress_bar.set_description(f"{epoch=:02d}, {loss=:.4f}, step={self.step:05d}")

                # Every n steps, evaluate as well
                if self.step % self.args.log_every_n_steps == 0:
                    self.log_samples()

        if self.args.use_wandb:
            wandb.finish()

        return self.model

In [7]:
# Run the train!
args = VAEArgs(use_wandb=False, epochs=5)
trainer = VAETrainer(args)
vae = trainer.train()

  0%|          | 0/118 [00:00<?, ?it/s]

  0%|          | 0/118 [00:00<?, ?it/s]

  0%|          | 0/118 [00:00<?, ?it/s]

  0%|          | 0/118 [00:00<?, ?it/s]

  0%|          | 0/118 [00:00<?, ?it/s]

Post traning checks

In [8]:
# Walk a 2D slice of the latent space: every other latent dim stays at 0,
# `dims` sweep -span..span (rows along dims[0], columns along dims[1])
@t.inference_mode()
def show_latent_grid(vae, dims=(0, 1), n=11, span=2.0):
    steps = t.linspace(-span, span, n)
    z = t.zeros(n, n, vae.latent_dim_size)
    z[..., dims[0]] = steps.unsqueeze(1)
    z[..., dims[1]] = steps
    imgs = vae.decoder(z.flatten(0, 1).to(device))   # (n*n, 1, 28, 28)
    display_data(imgs.cpu(), nrows=n, title=f"latent dims {dims}, {-span} to {span}")

show_latent_grid(vae)
show_latent_grid(vae, dims=(2, 3))   # with 5 latent dims, other pairs are worth a look


In [9]:
# Where the encoder puts each digit: the mean latent vector of n test images along two latent dims,
# coloured by digit, with the holdout digits drawn as thumbnails at their own positions
import plotly.express as px
from PIL import Image

@t.inference_mode()
def show_latent_scatter(vae, dims=(0, 1), n=5000):
    _, testset = mnist.get_mnist(trainset_size=0, testset_size=n)
    imgs = t.stack([img for img, _ in testset]).to(device)
    labels = [str(label) for _, label in testset]   # strings, so plotly colours by digit rather than on a scale
    means = vae.encoder(imgs)[0].cpu()              # the encoder gives (mean, log sigma); the mean is where an image sits

    fig = px.scatter(
        x=means[:, dims[0]].numpy(), y=means[:, dims[1]].numpy(), color=labels,
        category_orders={"color": [str(d) for d in range(10)]},
        labels={"x": f"dim {dims[0]}", "y": f"dim {dims[1]}", "color": "digit"},
        width=700, height=700, title=f"Latent space, dims {dims}",
    )

    # Holdout digits as thumbnails, un-normalised back to 0-255
    holdout_means = vae.encoder(trainer.holdout)[0].cpu()
    pixels = ((trainer.holdout.cpu() * 0.3081 + 0.1307).clamp(0, 1) * 255).byte().squeeze(1).numpy()
    size = (means[:, dims[0]].max() - means[:, dims[0]].min()).item() / 15
    for img, z in zip(pixels, holdout_means):
        fig.add_layout_image(
            source=Image.fromarray(img), xref="x", yref="y",
            x=z[dims[0]].item(), y=z[dims[1]].item(),
            xanchor="right", yanchor="top", sizex=size, sizey=size,
        )
    fig.show()

show_latent_scatter(vae)


Save model

In [10]:
NAME = "mnist_vae_test_1"
DESCRIPTION = "VAE trained on MNIST"

In [11]:
# Save the model to weights/, as a .pt + .json pair like the app's own saves
import json
from dataclasses import asdict
from datetime import datetime
from pathlib import Path

WEIGHTS_DIR = Path("..") / "weights"   # running from training/
path = WEIGHTS_DIR / f"{NAME}.pt"
assert not path.exists(), f"{path} already exists, pick another NAME"

# The app rebuilds models/mnist_vae.py with its default sizes, so the weights must fit that
default_shapes = {k: v.shape for k, v in VAE().state_dict().items()}
assert all(default_shapes[k] == v.shape for k, v in vae.state_dict().items()), \
    "latent/hidden sizes differ from VAE's defaults, so the app couldn't load this"

info = {
    "name": NAME,
    "model": "mnist_vae",
    "saved": datetime.now().isoformat(timespec="seconds"),
    "trained": datetime.now().isoformat(timespec="seconds"),
    "description": DESCRIPTION,
    "parameters": {
        "total": sum(p.numel() for p in vae.parameters()),
        "trainable": sum(p.numel() for p in vae.parameters() if p.requires_grad),
        "layers": {n: list(p.shape) for n, p in vae.named_parameters()},
    },
    # Not "MNIST": the MNIST test and dream windows expect a classifier and would break on a VAE
    "dataset": "MNIST VAE",
    "args": asdict(args),
    "batches_per_epoch": len(trainer.trainloader),
    "results": {
        "final_accuracy": None,
        "best_accuracy": None,
        "accuracy_per_epoch": [],
        "final_loss": None,
        "loss_per_epoch": [],
        "loss_per_batch": [],
    },
    "environment": {"device": str(device), "torch": t.__version__},
}

WEIGHTS_DIR.mkdir(exist_ok=True)
t.save(vae.state_dict(), path)
path.with_suffix(".json").write_text(json.dumps(info, indent=2))
print(f"saved {path} and {path.with_suffix('.json').name}")


saved ..\weights\mnist_vae_test_1.pt and mnist_vae_test_1.json
